# Polymath

**Jane Street puzzle, June 2015** — [puzzle page](https://www.janestreet.com/puzzles/polymath-index/)

> **AI use:** This notebook was scaffolded by an AI assistant — the title, link, and puzzle description below were pulled from the Jane Street archive automatically.

## Puzzle

![](https://www.janestreet.com/puzzles/Jun15_puzzle.png)

Choose an _n_-omino (call it T) and place as many copies of it as you can on the 10-by-10 board above.

* An _n_-omino is a connected region of _n_ cells. You get to choose _n_.
* Rotations and reflections of T are allowed.
* Copies of T may not overlap with each other.
* Any given copy of T may not be placed over two cells of the same value.
* The score for each copy of T is the product of the cells it covers.
* Your total score is the sum of the scores for your T's.

What's the highest total score you can get?

## Solution

**Answer: $$20160$$** — four copies of a heptomino, each covering one of every value from 1 to 7,
so each copy scores $7! = 5040$.

Two different heptomino families both reach 20160, and each of them has exactly one packing that
does it.

> **AI use:** the cells below were built with an AI assistant, working from my three-step plan and
> my first n-omino generator. That generator grew shapes correctly but never slid them back to the
> origin, so it treated every translation of a shape as a new shape.

The plan:

1. Generate every n-omino up to size 7, grouped into **families** — shapes that are the same up to
   rotation, reflection and translation.
2. Use CP-SAT to find the best packing of the board for each family.
3. Take the best result.

**Why size 7 is the ceiling.** The board only uses the values 1 through 7, and a copy may not cover
the same value twice, so no copy can have more than 7 cells. The same rule caps every copy's score
at $7! = 5040$: a 7-cell copy must cover exactly one of each value, and a 6-cell copy does best by
skipping the 1, which multiplies to the same 5040.

### The board

Transcribed from the picture, plus the 4×4 sample grid from the example.

In [2]:
from ortools.sat.python import cp_model

BOARD = [
    [6, 7, 5, 2, 6, 4, 2, 1, 4, 4],
    [5, 4, 6, 7, 7, 2, 4, 2, 6, 1],
    [5, 1, 7, 3, 4, 5, 5, 4, 7, 4],
    [2, 7, 6, 5, 1, 1, 2, 4, 6, 1],
    [1, 5, 3, 5, 3, 5, 3, 4, 5, 1],
    [6, 1, 2, 3, 4, 4, 5, 7, 2, 3],
    [1, 6, 5, 3, 3, 6, 5, 1, 1, 7],
    [2, 1, 1, 2, 1, 7, 1, 3, 3, 3],
    [7, 4, 4, 6, 3, 4, 1, 1, 6, 2],
    [4, 6, 5, 6, 2, 3, 7, 2, 3, 6],
]

EXAMPLE_BOARD = [
    [1, 1, 2, 3],
    [2, 1, 4, 1],
    [1, 2, 1, 2],
    [2, 3, 2, 4],
]

### Shapes, orientations and families

A shape is a tuple of `(row, column)` cells. To decide whether two shapes are "the same" we need one
agreed-upon way to write each one down:

- **Slide** the shape so its top row and left column are both zero, then sort the cells. Two copies
  of one shape in different places now look identical.
- **Rotate and reflect.** The eight symmetries of a square come from negating the rows, negating the
  columns, and swapping the two axes, in every combination ($2 \times 2 \times 2 = 8$).
- **Canonical form.** Of the eight images, take the smallest tuple. Two shapes belong to the same
  family exactly when their canonical forms match, so the canonical form is the family's name.

A symmetric shape has fewer than eight distinct orientations: the square has one, the straight bar
has two, the L has all eight.

In [1]:
def normalized(cells):
    "Return the same shape slid so its top row and left column are 0, as a sorted tuple."
    smallest_row = min(row for row, column in cells)
    smallest_column = min(column for row, column in cells)
    slid_cells = []
    for row, column in cells:
        slid_cells.append((row - smallest_row, column - smallest_column))
    return tuple(sorted(slid_cells))


def transformed(shape, flip_rows, flip_columns, swap_axes):
    "Return the shape after optionally negating rows, negating columns, and swapping the axes."
    new_cells = []
    for row, column in shape:
        if flip_rows:
            row = -row
        if flip_columns:
            column = -column
        if swap_axes:
            row, column = column, row
        new_cells.append((row, column))
    return normalized(new_cells)


def all_orientations(shape):
    "Return every distinct shape reachable from this one by rotating and reflecting it."
    orientations = set()
    for flip_rows in (False, True):
        for flip_columns in (False, True):
            for swap_axes in (False, True):
                orientations.add(transformed(shape, flip_rows, flip_columns, swap_axes))
    return sorted(orientations)


def canonical_form(shape):
    "Return the one agreed-upon representative of this shape's family."
    # Python compares tuples element by element, so min() picks a consistent orientation.
    return min(all_orientations(shape))


def shape_picture(shape):
    "Return the shape drawn as lines of # and . characters."
    height = max(row for row, column in shape) + 1
    width = max(column for row, column in shape) + 1
    lines = []
    for row in range(height):
        line = ""
        for column in range(width):
            if (row, column) in shape:
                line += "#"
            else:
                line += "."
        lines.append(line)
    return lines


L_TROMINO = ((0, 0), (0, 1), (1, 0))
for orientation in all_orientations(L_TROMINO):
    print("\n".join(shape_picture(orientation)))
    print()
print("canonical form of the L-tromino:", canonical_form(L_TROMINO))

##
#.

##
.#

#.
##

.#
##

canonical form of the L-tromino: ((0, 0), (0, 1), (1, 0))


### Checker first

`problems_with` checks every rule on a candidate solution before any model exists: each copy is a
rotation or reflection of T, sits on the board, and covers no value twice; and no board cell is
covered by two copies. `score_of` is the puzzle's scoring rule. A solution is a list of copies, and
each copy is a list of the board cells it covers.

In [ ]:
def product_of(values):
    "Return the product of a list of numbers."
    result = 1
    for value in values:
        result = result * value
    return result


def score_of(placements, board):
    "Return the total score: the sum over copies of the product of the values each covers."
    total = 0
    for cells in placements:
        values = []
        for row, column in cells:
            values.append(board[row][column])
        total += product_of(values)
    return total


def problems_with(shape, placements, board):
    "Return a list of everything wrong with a candidate solution. Empty means it is legal."
    size = len(board)
    problems = []
    allowed_orientations = all_orientations(shape)
    times_covered = {}

    for index, cells in enumerate(placements):
        if normalized(cells) not in allowed_orientations:
            problems.append(
                f"copy {index} {cells} is not a rotation or reflection of T"
            )

        values = []
        for row, column in cells:
            on_board = 0 <= row < size and 0 <= column < size
            if not on_board:
                problems.append(f"copy {index} runs off the board at {(row, column)}")
                continue
            if (row, column) not in times_covered:
                times_covered[(row, column)] = 0
            times_covered[(row, column)] += 1
            values.append(board[row][column])

        # A set drops duplicates, so the sizes match exactly when every value is different.
        if len(set(values)) != len(values):
            problems.append(f"copy {index} covers a repeated value: {sorted(values)}")

    for cell, count in times_covered.items():
        if count > 1:
            problems.append(f"cell {cell} is covered by {count} copies")

    return problems

### Validate on the example

The example places the L-tromino three times on the sample grid for $38 = 24 + 8 + 6$. The three
copies below are read off the picture; the second and third are rotated and reflected versions of T,
which is exactly what the checker has to accept.

In [4]:
EXAMPLE_SHAPE = ((0, 0), (0, 1), (1, 0))
EXAMPLE_PLACEMENTS = [
    [(0, 2), (0, 3), (1, 2)],  # 2, 3, 4 -> 24
    [(2, 2), (2, 3), (3, 3)],  # 1, 2, 4 -> 8
    [(2, 0), (2, 1), (3, 1)],  # 1, 2, 3 -> 6
]

example_problems = problems_with(EXAMPLE_SHAPE, EXAMPLE_PLACEMENTS, EXAMPLE_BOARD)
assert len(example_problems) == 0, example_problems
assert score_of(EXAMPLE_PLACEMENTS, EXAMPLE_BOARD) == 38

# A copy that covers two 1s must be rejected, and so must one that is not an L.
BAD_PLACEMENTS = [[(0, 0), (0, 1), (1, 0)], [(3, 0), (3, 1), (3, 2)]]
for problem in problems_with(EXAMPLE_SHAPE, BAD_PLACEMENTS, EXAMPLE_BOARD):
    print("expected problem:", problem)

print("example scores", score_of(EXAMPLE_PLACEMENTS, EXAMPLE_BOARD))

expected problem: copy 0 covers a repeated value: [1, 1, 2]
expected problem: copy 1 [(3, 0), (3, 1), (3, 2)] is not a rotation or reflection of T
expected problem: copy 1 covers a repeated value: [2, 2, 3]
example scores 38


### Generate the families

Grow the shapes one cell at a time from the previous size, normalizing every result and keeping
them in a set, so each *fixed* shape (rotations and reflections count as different) appears exactly
once. Then group the fixed shapes by canonical form: each key is one family, and the list under it
holds every orientation that family is allowed to use on the board.

The counts of free polyominoes (1, 1, 2, 5, 12, 35, 108) and fixed polyominoes
(1, 2, 6, 19, 63, 216, 760) are well known, so they double as a test of the generator.

In [5]:
NEIGHBOR_STEPS = [(1, 0), (-1, 0), (0, 1), (0, -1)]


def fixed_polyominoes(size):
    "Return the set of every connected shape of this size, with rotations counted as different."
    if size == 1:
        return {((0, 0),)}
    shapes = set()
    for smaller_shape in fixed_polyominoes(size - 1):
        for row, column in smaller_shape:
            for row_step, column_step in NEIGHBOR_STEPS:
                new_cell = (row + row_step, column + column_step)
                if new_cell not in smaller_shape:
                    shapes.add(normalized(smaller_shape + (new_cell,)))
    return shapes


def polyomino_families(size):
    "Return a dictionary from each family's canonical form to the list of its orientations."
    families = {}
    for shape in fixed_polyominoes(size):
        family = canonical_form(shape)
        if family not in families:
            families[family] = []
        families[family].append(shape)
    for family in families:
        families[family].sort()
    return families


KNOWN_FREE_COUNTS = {1: 1, 2: 1, 3: 2, 4: 5, 5: 12, 6: 35, 7: 108}
KNOWN_FIXED_COUNTS = {1: 1, 2: 2, 3: 6, 4: 19, 5: 63, 6: 216, 7: 760}

print(" n  families  fixed shapes")
for size in range(1, 8):
    families = polyomino_families(size)
    fixed_count = 0
    for family, members in families.items():
        fixed_count += len(members)
        # The members grouped by canonical form must be exactly the family's orientations.
        assert members == all_orientations(family)
    assert len(families) == KNOWN_FREE_COUNTS[size]
    assert fixed_count == KNOWN_FIXED_COUNTS[size]
    print(f"{size:>2}  {len(families):>8}  {fixed_count:>12}")

 n  families  fixed shapes
 1         1             1
 2         1             2
 3         2             6
 4         5            19
 5        12            63
 6        35           216
 7       108           760


### The model: pack one family

For one family, list every legal placement of a single copy: every orientation, at every position
where it fits on the board and covers no value twice. Each legal placement gets a boolean variable,
each board cell gets an at-most-one constraint over the placements that cover it, and the score is
the sum of the chosen placements' products. This is weighted set packing, and CP-SAT proves
optimality on it directly.

The model builder is shared by two solvers: one maximizes the score, and one (used at the end)
fixes the score and enumerates every packing that reaches it.

In [6]:
def legal_placements(shape, board):
    "Return every (cells, score) for one copy of the family placed legally on the board."
    size = len(board)
    placements = []
    for orientation in all_orientations(shape):
        height = max(row for row, column in orientation) + 1
        width = max(column for row, column in orientation) + 1
        for top_row in range(size - height + 1):
            for left_column in range(size - width + 1):
                cells = []
                for row, column in orientation:
                    cells.append((top_row + row, left_column + column))
                values = []
                for row, column in cells:
                    values.append(board[row][column])
                # A set drops duplicates, so the sizes match exactly when every value is different.
                if len(set(values)) == len(values):
                    placements.append((cells, product_of(values)))
    return placements


def packing_model(shape, board):
    "Return (model, is_used, placements, total_score) for packing copies of this family."
    placements = legal_placements(shape, board)
    model = cp_model.CpModel()

    # is_used[index] is true when placements[index] is one of the copies on the board.
    is_used = []
    for index in range(len(placements)):
        is_used.append(model.new_bool_var(f"placement_{index}"))

    # Copies may not overlap: for each board cell, at most one placement covering it is used.
    placements_covering = {}
    for index, (cells, score) in enumerate(placements):
        for cell in cells:
            if cell not in placements_covering:
                placements_covering[cell] = []
            placements_covering[cell].append(is_used[index])
    for cell, candidates in placements_covering.items():
        model.add_at_most_one(candidates)

    score_terms = []
    for index, (cells, score) in enumerate(placements):
        score_terms.append(score * is_used[index])
    total_score = sum(score_terms)

    return model, is_used, placements, total_score


def best_packing(shape, board):
    "Return (score, placements) for the highest-scoring packing of copies of this family."
    model, is_used, placements, total_score = packing_model(shape, board)
    model.maximize(total_score)

    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 8
    status = solver.solve(model)
    # OPTIMAL means the solver proved no better packing exists, not just that it stopped looking.
    assert status == cp_model.OPTIMAL, solver.status_name(status)

    chosen = []
    for index, (cells, score) in enumerate(placements):
        if solver.value(is_used[index]) == 1:
            chosen.append(cells)
    return int(solver.objective_value), chosen


example_score, example_chosen = best_packing(EXAMPLE_SHAPE, EXAMPLE_BOARD)
assert problems_with(EXAMPLE_SHAPE, example_chosen, EXAMPLE_BOARD) == []
print("best L-tromino packing of the sample grid scores", example_score)

best L-tromino packing of the sample grid scores 38


### Solve every family

164 families in all. Every model is tiny, because the no-repeated-values rule throws out most
placements: a heptomino has only a handful of legal spots on the whole board.

In [ ]:
results = []  # one (score, size, family, chosen placements) per family

print(" n  families  best score  copies  best family")
for size in range(1, 8):
    families = polyomino_families(size)
    best_for_size = None
    for family in sorted(families):
        score, chosen = best_packing(family, BOARD)
        results.append((score, size, family, chosen))
        if best_for_size is None or score > best_for_size[0]:
            best_for_size = (score, family, chosen)
    best_score, best_family, best_chosen = best_for_size
    picture = " / ".join(shape_picture(best_family))
    print(
        f"{size:>2}  {len(families):>8}  {best_score:>10}  {len(best_chosen):>6}  {picture}"
    )


def score_of_result(result):
    "Return the score stored in a results entry, for sorting."
    return result[0]


results.sort(key=score_of_result, reverse=True)

print()
print("top families overall:")
for score, size, family, chosen in results[:6]:
    print(
        f"  {score:>6}  n={size}  {len(chosen)} copies  {' / '.join(shape_picture(family))}"
    )

 n  families  best score  copies  best family
 1         1         381     100  #
 2         1         818      45  ##
 3         2        2359      26  ## / #.
 4         5        6524      15  ### / #..
 5        12       13482      12  ##. / .## / .#.
 6        35       19200       7  ##.. / .### / ...#
 7       108       20160       4  ##### / ##...

top families overall:
   20160  n=7  4 copies  ##### / ##...
   20160  n=7  4 copies  #### / #... / ##..
   19200  n=6  7 copies  ##.. / .### / ...#
   17268  n=6  7 copies  ### / #.# / #..
   16680  n=6  6 copies  ### / ##. / .#.
   16380  n=6  5 copies  #### / .#.. / .#..


### The answer

Two heptomino families tie at 20160. Each is placed four times, every copy covers one of each value
1 through 7, and the checker signs off on both.

In [ ]:
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def show_packing(placements, board):
    "Print the board values, and beside them which copy (A, B, C, ...) covers each cell."
    size = len(board)
    letter_at = {}
    for index, cells in enumerate(placements):
        for cell in cells:
            letter_at[cell] = LETTERS[index]
    for row in range(size):
        values = []
        letters = []
        for column in range(size):
            values.append(str(board[row][column]))
            if (row, column) in letter_at:
                letters.append(letter_at[(row, column)])
            else:
                letters.append(".")
        print(" ".join(values) + "     " + " ".join(letters))


BEST_SCORE = results[0][0]
winners = []
for score, size, family, chosen in results:
    if score == BEST_SCORE:
        winners.append((family, chosen))

for family, chosen in winners:
    assert problems_with(family, chosen, BOARD) == [], problems_with(
        family, chosen, BOARD
    )
    assert score_of(chosen, BOARD) == BEST_SCORE
    print("\n".join(shape_picture(family)))
    print()
    show_packing(chosen, BOARD)
    for index, cells in enumerate(chosen):
        values = []
        for row, column in cells:
            values.append(BOARD[row][column])
        print(
            f"  copy {LETTERS[index]} covers {sorted(values)} -> {product_of(values)}"
        )
    print()

print("answer:", BEST_SCORE)

#####
##...

6 7 5 2 6 4 2 1 4 4     . . . . . . . . . .
5 4 6 7 7 2 4 2 6 1     . . . . . . . . . .
5 1 7 3 4 5 5 4 7 4     . . . C C . . . . .
2 7 6 5 1 1 2 4 6 1     C C C C C A A . . .
1 5 3 5 3 5 3 4 5 1     . . . . . A A B B .
6 1 2 3 4 4 5 7 2 3     . . . . . A . B B .
1 6 5 3 3 6 5 1 1 7     . . . . . A . . B .
2 1 1 2 1 7 1 3 3 3     . . . . . A . . B .
7 4 4 6 3 4 1 1 6 2     . . . . . D D . B .
4 6 5 6 2 3 7 2 3 6     . . D D D D D . . .
  copy A covers [1, 2, 3, 4, 5, 6, 7] -> 5040
  copy B covers [1, 2, 3, 4, 5, 6, 7] -> 5040
  copy C covers [1, 2, 3, 4, 5, 6, 7] -> 5040
  copy D covers [1, 2, 3, 4, 5, 6, 7] -> 5040

####
#...
##..

6 7 5 2 6 4 2 1 4 4     . . . . . . . . . .
5 4 6 7 7 2 4 2 6 1     . . . . . . . . . .
5 1 7 3 4 5 5 4 7 4     . . . . . . . . . .
2 7 6 5 1 1 2 4 6 1     . C C . . . A A A A
1 5 3 5 3 5 3 4 5 1     . C . . . . A . . .
6 1 2 3 4 4 5 7 2 3     . C C C C . A A . .
1 6 5 3 3 6 5 1 1 7     B B B B . . . . . .
2 1 1 2 1 7 1 3 3 3     B . . . D D . 

### Why 20160 is optimal, and how many ways there are

Every family's model finished `OPTIMAL`, the families are exhaustive for sizes 1 through 7, and a
copy of size 8 or more is impossible, so the best per-family optimum is the global optimum. No copy
can score more than 5040, so 20160 means all four copies are perfect.

Last step of the method: drop the objective, fix the score at 20160, and enumerate. Each winning
family has exactly one packing that reaches it. That is not surprising once you see how few legal
placements each family has on the whole board.

In [9]:
class PackingCounter(cp_model.CpSolverSolutionCallback):
    "Counts every packing the solver reports during enumeration."

    def __init__(self):
        super().__init__()
        self.count = 0

    def on_solution_callback(self):
        self.count += 1


def count_packings_scoring(shape, board, target_score):
    "Return how many different packings of this family score exactly target_score."
    model, is_used, placements, total_score = packing_model(shape, board)
    # The score becomes a constraint, since an objective silently disables enumeration.
    model.add(total_score == target_score)

    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True
    solver.parameters.num_workers = 1  # enumeration only works with a single worker
    counter = PackingCounter()
    status = solver.solve(model, counter)
    assert status == cp_model.OPTIMAL, solver.status_name(status)
    return counter.count


for family, chosen in winners:
    legal_count = len(legal_placements(family, BOARD))
    packing_count = count_packings_scoring(family, BOARD, BEST_SCORE)
    print(
        f"{' / '.join(shape_picture(family))}: {legal_count} legal placements on the board,"
        f" {packing_count} packing scoring {BEST_SCORE}"
    )

##### / ##...: 4 legal placements on the board, 1 packing scoring 20160
#### / #... / ##..: 5 legal placements on the board, 1 packing scoring 20160
